# GPU training: report's required CPU-vs-GPU comparison, plus a real push past the CPU-only expression accuracy

This notebook has five parts.

**Part 1 (required by the brief)** runs the exact same `training/` scripts you already ran locally on CPU, on a Colab GPU, so Table 1's GPU column is an apples-to-apples comparison (same code/hyperparameters, only the hardware differs).

**Part 2 (optional, expression-accuracy push)** trains two additional expression models that were only practical with a GPU -- a higher-capacity MobileNetV2 (alpha=1.0) and a small CNN trained from scratch directly on FER2013 (not transfer learning) -- and evaluates every combination of all three expression models as a prediction-averaging ensemble, with and without test-time augmentation.

**Part 3 (fixes a real reported bug -- partially)** retrains the age/gender model with a crop-consistency fix found via a deep investigation into a real misprediction (a close-up selfie of a bearded young man scored 82.4 years -- see `training/analyze_age_bias.py` and the report's Limitations section for the full writeup). A second fix -- a counterfactual data augmentation randomly darkening the jaw of young training faces -- was also built and did measurably weaken the age-inflating effect, but caused a real regression: a bearded photo got misclassified as female, because age and gender share the same backbone and the augmentation degraded shared features. That fix is **not** in this notebook's default Part 3 -- only the crop-consistency fix is, which keeps gender reliable at the cost of not fully solving the original age issue. An optional cell at the end lets you reproduce the augmented (age-improved, gender-regressed) version too, for comparison, if you want to see both.

**Part 4 (age fix, round 3)** retries the beard augmentation at deliberately reduced strength (lower probability, more translucent), after Part 3's deployed model was confirmed via real-world testing to still have the facial-hair bias. Not guaranteed to work -- re-checks both age AND gender before you decide whether to deploy it, learning directly from Part 3b's regression.

**Part 5 (expression fix)** retrains both expression models with class weights informed by the deployed ensemble's *actual measured* per-class recall (Fear and Sad, not the naively-expected Surprise) instead of naive inverse-frequency weighting, after a real-world misread and a proper confusion-matrix analysis showed frequency-based weighting wasn't targeting the real weak classes.

**If you already ran earlier parts last time and downloaded those results:** you don't need to redo them. Just run cells 1-5 (setup/upload/data-prep) and skip to whichever part you need.

**Before running:** Runtime menu -> Change runtime type -> Hardware accelerator -> GPU (a free T4 is enough).

**Steps:**
1. Zip your local `training/` folder (just the scripts -- the data-prep cells re-download UTKFace/FER2013 straight from Hugging Face using Colab's own internet connection, so you do **not** need to include the local `data/` folder) and upload it below.
2. Run all cells top to bottom (or just cells 1-5 plus whichever part you need).
3. Download cells give you the result JSONs (for the report table) and the trained model files (to bring back to your laptop so they can be converted and deployed, the same way the CPU-trained models were).

In [ ]:
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU detected -- set Runtime > Change runtime type > GPU first.'

In [ ]:
!pip install -q datasets==5.0.0 "opencv-python-headless<5" scipy

In [ ]:
from google.colab import files
uploaded = files.upload()  # select your zipped training/ folder, e.g. training.zip

In [ ]:
import zipfile, pathlib
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('/content')
%cd /content/training

In [ ]:
!python prepare_utkface.py --build-manifest
!python prepare_fer2013.py --build-arrays
# Face-detects and crops every UTKFace training image the same way the deployed app
# crops a photo before classifying it (see prepare_utkface.py's module docstring for
# why). Takes about 10 minutes on Colab's CPU. Needed for Part 3; harmless if you're
# only doing Parts 1-2 (it just means age_gender_gpu.json would now be trained on the
# crop-consistent pipeline too, if you re-run Part 1 fresh -- if you already have Part
# 1's results from before, you don't need to touch this at all).
!python prepare_utkface.py --build-crops

## Part 1: required CPU-vs-GPU comparison
Same scripts, same hyperparameters as your local CPU run -- just `--tag gpu` instead of `--tag cpu`.

In [ ]:
!python train_age_gender.py --tag gpu --epochs 15 --finetune-epochs 10

In [ ]:
!python train_expression.py --tag gpu --epochs 20 --finetune-epochs 20

In [ ]:
from google.colab import files
files.download('results/age_gender_gpu.json')
files.download('results/expression_gpu.json')

## Part 2 (optional): pushing expression accuracy further
Two more expression models, only practical on GPU. Each takes roughly 10-30 minutes on a free Colab T4 -- if a cell is still running after ~40 minutes something is unusually slow (e.g. a shared/busy GPU); that's worth checking rather than assuming it's normal.

In [ ]:
# Higher-capacity MobileNetV2 (alpha=1.0 instead of 0.35). Saves to
# saved_models/expression_model_wide.keras -- does not overwrite the
# alpha=0.35 model from Part 1.
!python train_expression.py --tag gpu_wide --alpha 1.0 --epochs 15 --finetune-epochs 20

In [ ]:
# From-scratch CNN trained directly on FER2013's native 48x48 grayscale
# images -- not transfer learning. This is the architecture that was too
# slow to train on CPU (measured ~700s/epoch there); on GPU it should be a
# small fraction of that.
!python train_expression_scratch.py --tag gpu --epochs 100

In [ ]:
# Compares every individual model and every ensemble combination, with and
# without test-time augmentation, and reports the best one.
!python evaluate_expression_ensemble.py

In [ ]:
from google.colab import files

# Result summaries (small, always download these)
files.download('results/expression_gpu_wide.json')
files.download('results/expression_scratch_gpu.json')
files.download('results/expression_ensemble_comparison.json')

# Trained model files (bring these back to your laptop -- tell your
# assistant which combination won in the printed comparison above, and it
# will convert and deploy the winning one(s) to the app, the same way the
# CPU-trained models were converted).
files.download('saved_models/expression_model_wide.keras')
files.download('saved_models/expression_scratch_model.keras')

## Part 3: fixing the age-prediction bug (crop-consistency only -- the safe, deployed version)

Retrains age/gender with the crop-consistency fix only (no beard augmentation -- see the
notebook intro for why). This is the version actually deployed in the app. Takes roughly
10-20 minutes on a free T4 (plus the ~10 minutes `--build-crops` took in the data-prep
cell above, if you haven't run that yet this session). The diagnostic cell afterward now
checks gender as well as age -- added specifically because the skipped augmented version
broke gender without anything catching it before deployment.

In [ ]:
# Retrains age/gender with the crop-consistency fix only (--no-beard-aug). Saves to
# saved_models/age_gender_model.keras -- this is the version actually deployed.
!python train_age_gender.py --tag gpu_v3_no_beard_aug --epochs 15 --finetune-epochs 10 --no-beard-aug

In [ ]:
# Re-runs the bias/causal-test diagnostic against the model just trained above --
# now checks gender as well as age (added after the augmented version's gender
# regression), so this would have caught that problem before deployment.
!python analyze_age_bias.py

In [ ]:
from google.colab import files

# Result summary (for the report)
files.download('results/age_gender_gpu_v3_no_beard_aug.json')

# The retrained model (bring this back -- it replaces saved_models/age_gender_model.keras
# on your laptop, then gets converted and deployed the same way as before).
files.download('saved_models/age_gender_model.keras')

## Part 3b (optional): reproducing the augmented version, for comparison only

**Not recommended to deploy** -- this is the version with the gender regression,
included only so the report's before/after numbers are independently reproducible if you
want to verify them yourself. Skip this unless you specifically want that comparison.

In [ ]:
# Trains WITH the beard augmentation (the version with the gender regression).
# Back up the safe Part 3 model first, since training overwrites
# saved_models/age_gender_model.keras -- then restore it as the active file
# afterward, so Part 3's safe model stays the one that gets deployed.
import shutil
shutil.copy('saved_models/age_gender_model.keras', 'saved_models/age_gender_model_SAFE_v3.keras')

!python train_age_gender.py --tag gpu_v2_augmented_DO_NOT_DEPLOY --epochs 15 --finetune-epochs 10

shutil.copy('saved_models/age_gender_model.keras', 'saved_models/age_gender_model_augmented_DO_NOT_DEPLOY.keras')
shutil.copy('saved_models/age_gender_model_SAFE_v3.keras', 'saved_models/age_gender_model.keras')
print('Restored the safe (Part 3) model as saved_models/age_gender_model.keras.')
print('The augmented (regressed) model is saved separately as age_gender_model_augmented_DO_NOT_DEPLOY.keras.')

In [ ]:
from google.colab import files

# For reference/reproducibility only -- NOT the deployed model.
files.download('results/age_gender_gpu_v2_augmented_DO_NOT_DEPLOY.json')
files.download('saved_models/age_gender_model_augmented_DO_NOT_DEPLOY.keras')

## Part 4: age fix, round 3 (weaker beard augmentation, re-checked against gender)

Real-world testing after Part 3 shipped found the original age/facial-hair bias is still
present -- a bearded young face can still be over-predicted by well more than the model's
average error. That's expected: Part 3 deliberately shipped *without* the beard
augmentation (see the intro above), because round 1's augmentation broke gender.

This round tries a **deliberately weaker** version of the same augmentation --
`training/prepare_utkface.py`'s `BEARD_AUG_PROBABILITY` dropped from 0.25 to 0.12, and the
darkening strength from a near-opaque `[0.6, 0.95]` to a translucent `[0.25, 0.5]` -- on
the theory that a lighter counterfactual signal can still teach the model without
corrupting the shared backbone's gender-relevant features as badly. No code changes
needed here beyond what's already in the zip you're uploading; this just runs
`train_age_gender.py` **without** `--no-beard-aug`, so the (now weaker) augmentation is
active.

**This is not guaranteed to work** -- it might still regress gender, just less, or might
not meaningfully help age either. That's why the next cell re-checks **both** age and
gender before you decide what to deploy, exactly the check that was missing before round
1 shipped.

**Read the printed output of the diagnostic cell carefully and decide:**
- If gender's `pct_misclassified_female` on the jaw-patch test stays close to Part 3's
  baseline (roughly 0-10%) **and** the age jaw-vs-control effect shrank meaningfully
  compared to Part 3's numbers (report's Limitations section has those) -- this is a real
  improvement, worth deploying (tell your assistant to convert and deploy it).
- If gender's misclassification rate jumps well above baseline again -- this is round 1
  happening again, just less severely. Don't deploy it; keep Part 3's model, and report
  this attempt (with its real numbers) as a documented, unsuccessful further attempt in
  the Limitations section -- that's a legitimate, honest outcome, not a failure to hide.</cell id="PLACEHOLDER">

In [ ]:
# Retrains age/gender WITH the (now weaker) beard augmentation active -- no
# --no-beard-aug flag this time. Saves to saved_models/age_gender_model.keras,
# OVERWRITING Part 3's safe model in this Colab session (your laptop's copy is
# untouched until you actually download and replace it below).
!python train_age_gender.py --tag gpu_v4_beard_aug_round3 --epochs 15 --finetune-epochs 10

In [ ]:
# Re-runs the bias/causal-test diagnostic against the round-3 model just trained.
# Read BOTH the age section and the gender section printed below -- this is the
# check that must pass on gender before this model is worth deploying at all.
!python analyze_age_bias.py

In [ ]:
from google.colab import files

# Result summary (for the report -- keep this even if you decide not to deploy the
# model, since an honestly-reported unsuccessful attempt is real content for the
# Limitations section).
files.download('results/age_gender_gpu_v4_beard_aug_round3.json')

# Only bring this back to your laptop if the diagnostic cell above looked good on
# BOTH age and gender. If gender regressed again, skip this download entirely --
# your laptop should keep Part 3's model as the deployed one.
files.download('saved_models/age_gender_model.keras')

## Part 5: expression fix -- recall-informed class weights

Real-world testing surfaced a misread (Surprise predicted as Happy). Checking the actual
confusion matrix of the currently-deployed ensemble against the full FER2013 test set
(2,910 images) showed that guess was only half right: Surprise itself is one of the
model's *better* classes (73.9% recall). The genuinely weak classes are **Fear (51.5%)**
and **Sad (51.4%)** -- and notably, Sad has *more* training examples (3,750) than Angry
(3,116) or Surprise (2,094), so its weakness isn't a data-quantity problem. Both scripts
already applied naive inverse-frequency class weighting (which mainly upweights the
smallest class, Surprise) and it wasn't enough, which is consistent with that: the
bottleneck is visual separability (Fear/Sad/Angry are a well-documented "hard triad" in
the FER2013 literature), not sample count.

`train_expression.py` and `train_expression_scratch.py` now compute class weights from
the *observed per-class recall* of the currently-deployed ensemble instead of raw
frequency -- Fear and Sad get upweighted the most (~1.23x), Happy (already at 85.8%
recall) gets downweighted (~0.74x). This directly targets the classes the evidence says
are actually struggling. No flags needed -- both scripts pick this up automatically.

**Realistic expectation:** this is a legitimate, evidence-based attempt, not a guaranteed
win. Published FER2013 results in the 65-75% range are typical for models this size;
closing Fear/Sad's gap by a few points would be a genuine, honestly-reportable
improvement, not a full fix -- the confusion matrix afterward is what tells you whether it
actually helped, not just the headline accuracy number.

In [ ]:
# Retrains both expression models that feed the deployed ensemble, now with
# recall-informed class weights (automatic -- no flags needed). Overwrites
# saved_models/expression_model_wide.keras and expression_scratch_model.keras.
!python train_expression.py --tag gpu_wide_v2_recall_weighted --alpha 1.0 --epochs 15 --finetune-epochs 20
!python train_expression_scratch.py --tag gpu_v2_recall_weighted --epochs 100

In [ ]:
# Re-compares every combination, and now also prints a full confusion matrix and
# per-class recall for the winning combination -- compare Fear/Sad's recall here
# against the baseline (51.5% / 51.4%) to see whether this actually helped the
# classes it targeted, not just the headline accuracy number.
!python evaluate_expression_ensemble.py

In [ ]:
from google.colab import files

# Result summaries (for the report -- keep these regardless of outcome).
files.download('results/expression_gpu_wide_v2_recall_weighted.json')
files.download('results/expression_scratch_gpu_v2_recall_weighted.json')
files.download('results/expression_ensemble_comparison.json')

# Trained model files -- bring these back and tell your assistant the new best
# combination's accuracy and per-class recall from the printed output above,
# so it can compare against the deployed baseline (67.60% overall, Fear 51.5%,
# Sad 51.4%) before deciding whether to convert and deploy this version.
files.download('saved_models/expression_model_wide.keras')
files.download('saved_models/expression_scratch_model.keras')